 ## Cart2Insights — Step 6: Store Cleaned Data in SQL

## Objective

Create the database

Create tables based on the ER diagram

Load cleaned data into SQL

Apply primary and foreign key constraints

Verify the data after loading



Import Libraries

In [1]:
import numpy as np
import pandas as pd
import sqlite3
import os

Load cleaned datasets

In [2]:
customers = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/cleaned/customers_cleaned.csv')
geolocation = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/cleaned/geolocation_cleaned.csv')
order_items = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/cleaned/order_items_cleaned.csv')
order_payments = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/cleaned/order_payments_cleaned.csv')
order_reviews = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/cleaned/order_reviews_cleaned.csv')
orders = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/cleaned/orders_cleaned.csv')
products = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/cleaned/products_cleaned.csv')
sellers = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/cleaned/sellers_cleaned.csv')
category_translation = pd.read_csv('/content/drive/MyDrive/cart_to_insight/data/cleaned/category_translation_cleaned.csv')

In [3]:
# Create a connection to the SQLite database inside the Google Drive project folder
# This will create the file 'cart_to_insight.db' if it doesn't exist
db_dir = '/content/drive/MyDrive/cart_to_insight/data'
os.makedirs(db_dir, exist_ok=True)
db_path = os.path.join(db_dir, 'cart_to_insight.db')

conn = sqlite3.connect(db_path)
cursor = conn.cursor()
print(f"Database '{db_path}' created and connected successfully.")

Database '/content/drive/MyDrive/cart_to_insight/data/cart_to_insight.db' created and connected successfully.


In [4]:
# Enable foreign key support in SQLite
conn.execute("PRAGMA foreign_keys = ON;")

# Drop tables to recreate with corrected schema
conn.execute("PRAGMA foreign_keys = OFF;")
tables_to_drop = ['customers', 'geolocation', 'products', 'sellers', 'category_translation', 'orders', 'order_items', 'order_payments', 'order_reviews']
for t in tables_to_drop:
    conn.execute(f"DROP TABLE IF EXISTS {t};")
conn.commit()
conn.execute("PRAGMA foreign_keys = ON;")

# Define the SQL schema based on the Olist ER diagram relations
# 1. Geolocation Table
conn.execute('''
CREATE TABLE IF NOT EXISTS geolocation (
    geolocation_zip_code_prefix INTEGER,
    geolocation_lat REAL,
    geolocation_lng REAL,
    geolocation_city TEXT,
    geolocation_state TEXT
);
''')

# 2. Customers Table
conn.execute('''
CREATE TABLE IF NOT EXISTS customers (
    customer_id TEXT PRIMARY KEY,
    customer_unique_id TEXT,
    customer_zip_code_prefix INTEGER,
    customer_city TEXT,
    customer_state TEXT
);
''')

# 3. Sellers Table
conn.execute('''
CREATE TABLE IF NOT EXISTS sellers (
    seller_id TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city TEXT,
    seller_state TEXT
);
''')

# 4. Products Table (Corrected spelling for length columns)
conn.execute('''
CREATE TABLE IF NOT EXISTS products (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT,
    product_name_length REAL,
    product_description_length REAL,
    product_photos_qty REAL,
    product_weight_g REAL,
    product_length_cm REAL,
    product_height_cm REAL,
    product_width_cm REAL
);
''')

# 5. Orders Table
conn.execute('''
CREATE TABLE IF NOT EXISTS orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT,
    order_status TEXT,
    order_purchase_timestamp TEXT,
    order_approved_at TEXT,
    order_delivered_carrier_date TEXT,
    order_delivered_customer_date TEXT,
    order_estimated_delivery_date TEXT,
    FOREIGN KEY (customer_id) REFERENCES customers (customer_id)
);
''')

# 6. Order Items Table
conn.execute('''
CREATE TABLE IF NOT EXISTS order_items (
    order_id TEXT,
    order_item_id INTEGER,
    product_id TEXT,
    seller_id TEXT,
    shipping_limit_date TEXT,
    price REAL,
    freight_value REAL,
    PRIMARY KEY (order_id, order_item_id),
    FOREIGN KEY (order_id) REFERENCES orders (order_id),
    FOREIGN KEY (product_id) REFERENCES products (product_id),
    FOREIGN KEY (seller_id) REFERENCES sellers (seller_id)
);
''')

# 7. Order Payments Table
conn.execute('''
CREATE TABLE IF NOT EXISTS order_payments (
    order_id TEXT,
    payment_sequential INTEGER,
    payment_type TEXT,
    payment_installments INTEGER,
    payment_value REAL,
    FOREIGN KEY (order_id) REFERENCES orders (order_id)
);
''')

# 8. Order Reviews Table
conn.execute('''
CREATE TABLE IF NOT EXISTS order_reviews (
    review_id TEXT,
    order_id TEXT,
    review_score INTEGER,
    review_comment_title TEXT,
    review_comment_message TEXT,
    review_creation_date TEXT,
    review_answer_timestamp TEXT,
    FOREIGN KEY (order_id) REFERENCES orders (order_id)
);
''')

# 9. Category Translation Table
conn.execute('''
CREATE TABLE IF NOT EXISTS category_translation (
    product_category_name TEXT PRIMARY KEY,
    product_category_name_english TEXT
);
''')

print("Tables created with correct schemas successfully.")

Tables created with correct schemas successfully.


In [5]:
# Temporarily disable foreign keys to safely clear existing data
conn.execute("PRAGMA foreign_keys = OFF;")
tables = ['customers', 'geolocation', 'products', 'sellers', 'category_translation', 'orders', 'order_items', 'order_payments', 'order_reviews']
for table in tables:
    conn.execute(f"DELETE FROM {table};")
conn.commit()

# Re-enable foreign key checks
conn.execute("PRAGMA foreign_keys = ON;")

# Populate SQL tables in the correct order to avoid Foreign Key violations
# 1. Parent tables first (no foreign key dependencies)
customers.to_sql('customers', conn, if_exists='append', index=False)
geolocation.to_sql('geolocation', conn, if_exists='append', index=False)
products.to_sql('products', conn, if_exists='append', index=False)
sellers.to_sql('sellers', conn, if_exists='append', index=False)
category_translation.to_sql('category_translation', conn, if_exists='append', index=False)

# 2. Dependent tables next
orders.to_sql('orders', conn, if_exists='append', index=False)

# 3. Tables depending on orders, products, or sellers
order_items.to_sql('order_items', conn, if_exists='append', index=False)
order_payments.to_sql('order_payments', conn, if_exists='append', index=False)
order_reviews.to_sql('order_reviews', conn, if_exists='append', index=False)

print("Data loaded successfully into all tables in the correct relational hierarchy!")

# Verification: Check row count in each table to verify completion
print("\n--- Verification: Row Counts in SQL ---")
for table in tables:
    count = conn.execute(f"SELECT COUNT(*) FROM {table};").fetchone()[0]
    print(f"Table '{table}': {count} rows")

Data loaded successfully into all tables in the correct relational hierarchy!

--- Verification: Row Counts in SQL ---
Table 'customers': 99441 rows
Table 'geolocation': 738332 rows
Table 'products': 32951 rows
Table 'sellers': 3095 rows
Table 'category_translation': 71 rows
Table 'orders': 99441 rows
Table 'order_items': 112650 rows
Table 'order_payments': 103886 rows
Table 'order_reviews': 99224 rows
